# Setup and logging

In [0]:
dbutils.widgets.text("catalog", "workspace", "Catalog name")
dbutils.widgets.text("db_schema", "dav_project", "Schema (database) name")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA = dbutils.widgets.get("db_schema")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"USE {CATALOG}.{SCHEMA}")

print(f"Using {CATALOG}.{SCHEMA}")

Using workspace.dav_project


## Bronze tables

In [0]:
BRONZE_ENTITY_DDL = """
CREATE TABLE IF NOT EXISTS {catalog}.{schema}.bronze_{entity} (
  record_key      STRING    NOT NULL,
  source_repo     STRING    NOT NULL,
  entity_type     STRING    NOT NULL,
  raw_json        STRING    NOT NULL,
  api_params      STRING,
  load_type       STRING    NOT NULL,
  batch_id        STRING    NOT NULL,
  run_id          STRING    NOT NULL,
  load_timestamp  TIMESTAMP NOT NULL
) USING DELTA
"""

for entity in ["commits", "issues", "pulls"]:
    spark.sql(BRONZE_ENTITY_DDL.format(catalog=CATALOG, schema=SCHEMA, entity=entity))
    print(f"ok: bronze_{entity}")

ok: bronze_commits
ok: bronze_issues
ok: bronze_pulls


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.bronze_quarantine (
  source_repo     STRING,
  entity_type     STRING,
  raw_record      STRING,
  reason          STRING,
  batch_id        STRING,
  run_id          STRING,
  load_timestamp  TIMESTAMP
) USING DELTA
""")
print("ok: bronze_quarantine")

ok: bronze_quarantine


## Silver tables

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_commit (
  sha                 STRING    NOT NULL,
  repo                STRING    NOT NULL,
  author_key          STRING,
  committer_key       STRING,
  committed_at         TIMESTAMP,
  message_length       INT,
  is_merge_commit       BOOLEAN,
  load_timestamp        TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_issue (
  issue_id             STRING    NOT NULL,
  repo                 STRING    NOT NULL,
  number                INT,
  state                 STRING,
  author_key            STRING,
  created_at            TIMESTAMP,
  updated_at            TIMESTAMP,
  closed_at             TIMESTAMP,
  comment_count          INT,
  time_to_close_hours    DOUBLE,
  load_timestamp         TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_pull_request (
  pr_id                STRING    NOT NULL,
  repo                 STRING    NOT NULL,
  number                INT,
  state                 STRING,
  author_key            STRING,
  created_at            TIMESTAMP,
  updated_at            TIMESTAMP,
  closed_at             TIMESTAMP,
  merged_at             TIMESTAMP,
  is_merged              BOOLEAN,
  is_draft                BOOLEAN,
  base_branch            STRING,
  time_to_merge_hours     DOUBLE,
  load_timestamp          TIMESTAMP NOT NULL
) USING DELTA
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.silver_quarantine (
  entity_type      STRING,
  record_key        STRING,
  repo               STRING,
  reason              STRING,
  raw_json            STRING,
  batch_id             STRING,
  run_id               STRING,
  load_timestamp        TIMESTAMP
) USING DELTA
""")

print("ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine")

ok: silver_commit, silver_issue, silver_pull_request, silver_quarantine


## Log table

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{SCHEMA}.pipeline_execution_logs (
  run_id             STRING    NOT NULL,
  layer               STRING    NOT NULL,
  entity                STRING,
  parameters              STRING,
  start_time               TIMESTAMP NOT NULL,
  end_time                 TIMESTAMP,
  status                     STRING,
  rows_inserted               BIGINT,
  rows_updated                 BIGINT,
  rows_quarantined               BIGINT,
  error_message                    STRING
) USING DELTA
""")
print("ok: pipeline_execution_logs")

ok: pipeline_execution_logs


## Small helper functions

In [0]:
import uuid
import json
from datetime import datetime, timezone
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, LongType


def new_run_id() -> str:
    return str(uuid.uuid4())


def now_utc():
    return datetime.now(timezone.utc)

LOG_SCHEMA = StructType([
    StructField("run_id", StringType(), False),
    StructField("layer", StringType(), False),
    StructField("entity", StringType(), True),
    StructField("parameters", StringType(), True),
    StructField("start_time", TimestampType(), False),
    StructField("end_time", TimestampType(), True),
    StructField("status", StringType(), True),
    StructField("rows_inserted", LongType(), True),
    StructField("rows_updated", LongType(), True),
    StructField("rows_quarantined", LongType(), True),
    StructField("error_message", StringType(), True),
])

def log_run(run_id, layer, entity, parameters: dict, start_time, end_time,
            status, rows_inserted=0, rows_updated=0, rows_quarantined=0,
            error_message=None):
    """Writes exactly one row to pipeline_execution_logs. Never raises."""
    row = [(
        run_id, layer, entity, json.dumps(parameters),
        start_time, end_time, status,
        int(rows_inserted), int(rows_updated), int(rows_quarantined),
        error_message,
    )]
    # cols = ["run_id", "layer", "entity", "parameters", "start_time", "end_time",
    #         "status", "rows_inserted", "rows_updated", "rows_quarantined", "error_message"]
    df = spark.createDataFrame(row, schema=LOG_SCHEMA)
    df.write.format("delta").mode("append").saveAsTable(
        f"{CATALOG}.{SCHEMA}.pipeline_execution_logs"
    )
    print(f"[log] {layer} / {entity} -> {status} "
          f"(inserted={rows_inserted}, updated={rows_updated}, quarantined={rows_quarantined})")


def get_merge_metrics(full_table_name: str) -> dict:
    """Reads the row counts from the most recent MERGE on a Delta table."""
    hist = spark.sql(f"DESCRIBE HISTORY {full_table_name} LIMIT 1").collect()[0]
    metrics = hist["operationMetrics"] or {}
    return {
        "inserted": int(metrics.get("numTargetRowsInserted", 0)),
        "updated": int(metrics.get("numTargetRowsUpdated", 0)),
    }


print("helpers ready: new_run_id, now_utc, log_run, get_merge_metrics")

helpers ready: new_run_id, now_utc, log_run, get_merge_metrics
